# 6주차 ③ 재현성과 실험 관리 — 실습 8~11  〔빈칸본〕

> **3교시 노트북입니다.** 5주차의 `models/mlp_fashion.pt` 가 필요합니다.
> 빈칸은 모두 **8곳 · 18줄** 입니다 —
> 실습 8에 1곳(5줄), 실습 9에 2곳(2줄), 실습 10에 3곳(8줄), 실습 11에 2곳(3줄).
>
> 다 채운 노트북은 `10_experiment_tracking.ipynb` 로 저장합니다.

> **오늘의 질문**: *"다음에 또 이 결과를 만들 수 있는가?"*
>
> ```
>    월요일 : 정확도 88.4% 가 나왔다! 스크린샷을 찍어 뒀다
>    목요일 : 같은 코드를 다시 돌렸다 -> 86.9%
>             ...어느 쪽이 진짜지? 내가 뭘 바꿨더라?
> ```
>
> 지금은 학습이 30초라 와닿지 않습니다. **9주차 LoRA, 12주차 BERT 파인튜닝은 수십 분**입니다.
> 그때 이 습관이 없으면 실험을 통제하지 못합니다.

---

## 실습 8 — 시드 고정 전/후 (6분)

| 결과가 달라지는 원인 | 해결 |
|---|---|
| 가중치 **초기화**가 무작위 | `torch.manual_seed()` |
| `DataLoader` 의 **셔플** 순서 | 같은 시드 |
| 드롭아웃이 끄는 뉴런 | 같은 시드 |
| GPU 의 **비결정적 연산** | 완전히는 못 막는다 ★ |

> **시드를 고정해도 GPU 에서는 완전히 같지 않을 수 있습니다** ★ (출제 지점).
> 병렬 연산의 덧셈 순서가 실행마다 달라질 수 있기 때문입니다.
> `torch.use_deterministic_algorithms(True)` 로 강제할 수 있지만 **느려집니다.**
> 실무에서는 **"거의 같으면 충분"** 으로 두고, 대신 **조건을 전부 기록**합니다.

In [ ]:
# 셀 1 — 시드를 안 고정하면
import torch, torch.nn as nn, random, numpy as np

def build():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

print("[시드 고정 안 함]")
for i in range(2):
    m = build()
    print(f"  {i+1}회차 첫 가중치 : {m[1].weight[0, :3].tolist()}")

In [ ]:
# 셀 2 — 시드 고정 함수
# ───── 실습 8 · 빈칸 ① (5줄) ─────
# 힌트 : 무작위성의 출처 네 곳을 한 번에 고정한다. random / numpy / torch / torch.cuda (4줄)
# 확인 : 아래 두 줄의 가중치가 완전히 같게 나온다






print("[시드 고정]")
for i in range(2):
    set_seed(42)
    m = build()
    print(f"  {i+1}회차 첫 가중치 : {m[1].weight[0, :3].tolist()}")

> **관찰 포인트 ★**: 위쪽은 두 번의 값이 다르고, 아래쪽은 **완전히 같습니다.**
> 이 네 줄짜리 함수가 **모든 실험 노트북의 맨 위에** 들어가야 합니다.

> **포인트**: `set_seed` 를 `seed_utils.py` 로 빼서 매 노트북에서
> `from seed_utils import set_seed` 로 불러 쓰면 편합니다. 배포 파일에 들어 있습니다.

> **함정**: 시드는 **모델을 만들기 직전**에 불러야 합니다. 노트북 맨 위에서 한 번만 부르고
> 셀을 여기저기 실행하면, 실행 순서에 따라 값이 달라집니다.

---

## 실습 9 — `state_dict` 저장·불러오기 (8분)

```
   torch.save(model, "m.pt")               ✗ 모델 객체 통째로
        -> 클래스 정의 경로에 의존한다. 다른 PC·다른 폴더 구조에서 깨진다

   torch.save(model.state_dict(), "m.pt")  ○ 가중치만  ★ 이걸 쓴다
        -> 구조는 코드가 갖고 있으므로, 코드 + 가중치면 어디서든 복원된다
```

> **`state_dict` 만 저장합니다** ★ (출제 지점).
> 모델 전체 저장은 *"이 클래스가 이 경로에 있다"* 는 가정을 함께 저장하는 셈이라 깨지기 쉽습니다.
> 5주차에 `state_dict()` 를 쓴 이유가 이것입니다.

In [ ]:
# 셀 3 — 5주차 가중치를 불러온다
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

device = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
test = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
test_loader = DataLoader(test, batch_size=256)

class MLP(nn.Module):
    """★ 5주차 07_mlp_fashionmnist.ipynb 의 MLP 와 **글자 그대로 같아야** 한다.

    state_dict 는 층 이름(net.0.weight ...)과 shape 으로 맞추므로,
    self.flatten / self.net 으로 나눈 구조까지 그대로 옮겨 온다.
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(self.flatten(x))


def accuracy(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t


fresh = MLP().to(device)
print(f"학습 안 한 새 모델 : {accuracy(fresh)*100:5.2f}%   <- 찍기 수준")

loaded = MLP().to(device)

# ───── 실습 9 · 빈칸 ① (1줄) ─────
# 힌트 : 5주차 가중치 파일을 불러와 새 모델에 넣는다. map_location=device 를 잊지 말 것 (1줄)
# 확인 : 5주차와 같은 86% 대가 나온다


print(f"5주차 가중치 로드  : {accuracy(loaded)*100:5.2f}%   <- 5주차 결과와 같아야 한다")

> **관찰 포인트 ★** — 실측값입니다.
>
> ```
> 학습 안 한 새 모델 : 10.09%   <- 찍기 수준
> 5주차 가중치 로드  : 86.20%   <- 5주차 결과와 같아야 한다
> ```
>
> **86.20% 는 5주차 실습 10에서 나온 바로 그 숫자입니다.**
> 코드는 새로 썼고 모델도 방금 만들었는데, **가중치 파일 하나를 넣었더니 성능이 그대로 따라왔습니다.**
> 이게 9주차 사전학습 모델의 원리이기도 합니다.

> **함정 ★**: 구조가 조금이라도 다르면 `Missing key(s)` / `size mismatch` 오류가 납니다.
> **`state_dict` 는 층 이름과 shape 으로 맞춥니다.** 오류 메시지가 어느 층인지 알려 주니 끝까지 읽으세요.

> **포인트**: `map_location=device` 는 **GPU 에서 저장한 것을 CPU 에서 열 때** 필요합니다.
> 실습실과 개인 노트북을 오갈 때 매번 씁니다.

> **막히면**: `models/mlp_fashion.pt` 가 없으면 배포본을 쓰세요(교수 안내).

In [ ]:
# 셀 4 — state_dict 안에는 무엇이 들어 있나
# ───── 실습 9 · 빈칸 ② (1줄) ─────
# 힌트 : 불러온 모델의 state_dict 를 꺼낸다
# 확인 : 항목 6개(weight·bias 각 3쌍)가 층 이름과 shape 으로 출력된다


print(f"항목 {len(sd)}개\n")
for k, v in sd.items():
    print(f"  {k:20s} {tuple(v.shape)}")
print("\n-> 층 이름 + shape 의 사전일 뿐입니다. 구조 정보는 없습니다.")

---

## 실습 10 — 체크포인트로 학습 재개 ★★ (10분)

```
   학습 3시간째에 노트북이 꺼졌다. 가중치는 저장돼 있다.
   그런데 이어서 돌리면?

     ✗ 옵티마이저가 처음 상태로 돌아간다
         Momentum 이 쌓아 둔 관성, Adam 이 쌓아 둔 통계가 전부 0
         -> 손실이 한 번 튀었다가 다시 안정된다
     ✗ 지금이 몇 epoch 인지 모른다 -> 스케줄러도 처음부터
```

> **체크포인트에는 모델 + 옵티마이저 + epoch 를 함께 담습니다** ★ (출제 지점).
> 옵티마이저 상태를 빼면 재개 직후 손실이 튑니다.

In [ ]:
# 셀 5 — 5 epoch 만 돌리고 체크포인트 저장
train_full = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
train_loader = DataLoader(Subset(train_full, range(10000)), batch_size=128, shuffle=True)

set_seed(42)
model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
lossfn = nn.CrossEntropyLoss()

def train_epochs(model, opt, start, end):
    for epoch in range(start, end):
        model.train(); run = n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item()*xb.size(0); n += xb.size(0)
        print(f"  epoch {epoch+1} | loss {run/n:.4f}")

train_epochs(model, opt, 0, 5)
os.makedirs("models", exist_ok=True)

# ───── 실습 10 · 빈칸 ① (3줄) ─────
# 힌트 : 체크포인트에 담을 세 가지 — epoch, 모델 가중치, 옵티마이저 상태 (3줄)
# 확인 : models/checkpoint_epoch5.pt 가 만들어진다



           "models/checkpoint_epoch5.pt")
print("체크포인트 저장 완료")

> ### ★ 여기서 **커널 → 다시 시작(Restart Kernel)** 하세요.
>
> *"학습이 끊긴 상황"* 을 실제로 만들어야 다음 셀의 의미가 삽니다.
> 재시작한 뒤에는 **셀 1 ~ 5 를 다시 실행**해야 `import` 와 `MLP` 클래스가 살아납니다.
> *(`Run All Above` 를 쓰면 편합니다.)*

In [ ]:
# 셀 6 — 커널 재시작 후, 이어서 학습
ckpt = torch.load("models/checkpoint_epoch5.pt", map_location=device)

model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

# ───── 실습 10 · 빈칸 ② (3줄) ─────
# 힌트 : 모델 가중치와 옵티마이저 상태를 복원하고, 몇 epoch 부터인지 꺼낸다 (3줄)
# 확인 : epoch 5 부터 재개된다




print(f"epoch {start} 부터 재개")
train_epochs(model, opt, start, start + 3)

In [ ]:
# 셀 7 — 옵티마이저 상태를 빼면 어떻게 되나 (비교)
print("[옵티마이저 상태를 복원한 경우]")
m1 = MLP().to(device); o1 = torch.optim.SGD(m1.parameters(), lr=0.1, momentum=0.9)
m1.load_state_dict(ckpt["model"]); o1.load_state_dict(ckpt["optimizer"])
train_epochs(m1, o1, 5, 6)

print("\n[옵티마이저 상태를 빼먹은 경우]")
m2 = MLP().to(device); o2 = torch.optim.SGD(m2.parameters(), lr=0.1, momentum=0.9)
m2.load_state_dict(ckpt["model"])              # ★ 옵티마이저는 복원하지 않는다
train_epochs(m2, o2, 5, 6)

print("\n-> 아래쪽 손실이 더 높으면, 그게 관성을 잃은 대가입니다")

> **관찰 포인트 ★★** — 실측값입니다.
>
> ```
> epoch 5 (끊기기 직전)          loss 0.4029
> epoch 6 (체크포인트로 재개)     loss 0.3425   <- 자연스럽게 이어진다
>
> [옵티마이저 상태를 복원한 경우]  epoch 6 | loss 0.3524
> [옵티마이저 상태를 빼먹은 경우]  epoch 6 | loss 0.3554
> ```
>
> 재개 첫 epoch 의 손실이 **5 epoch 째와 자연스럽게 이어집니다.**
>
> ⚠️ **솔직하게 말하면, 아래 두 줄의 차이는 작습니다**(0.3524 vs 0.3554).
> 우리 설정은 **학습이 짧고 이미 안정 구간**이라 관성을 잃어도 금방 따라잡습니다.
> **방향은 항상 같습니다** — 옵티마이저를 빼면 손실이 더 높습니다.
> 학습이 길거나 Adam 처럼 내부 통계를 많이 쌓는 옵티마이저일수록 차이가 커집니다.

In [ ]:
# 셀 8 — best_model.pt : 검증 성능이 좋아질 때만 덮어쓴다
val_loader = DataLoader(Subset(train_full, range(50000, 52000)), batch_size=256)

def val_loss(m):
    m.eval(); s = n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            s += lossfn(m(xb), yb).item() * xb.size(0); n += xb.size(0)
    return s / n

set_seed(42)
model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
best = float("inf")

for epoch in range(8):
    train_epochs(model, opt, epoch, epoch + 1)
    v = val_loss(model)
    mark = ""

# ───── 실습 10 · 빈칸 ③ (2줄) ─────
# 힌트 : 검증 손실이 지금까지보다 좋아졌을 때만 best 를 갱신하고 가중치를 저장한다 (2줄)
# 확인 : "best_model.pt 갱신" 이 앞쪽 epoch 에서 몇 번 찍히다가 멈춘다



    print(f"    검증 loss {v:.4f}{mark}")

print(f"\n가장 좋았던 검증 손실 : {best:.4f}")

> **핵심 메시지 ★★** — 실측값을 보세요.
>
> ```
> epoch 1 | 훈련 0.9087 | 검증 0.7527  <- best_model.pt 갱신
> epoch 2 | 훈련 0.5344 | 검증 0.4735  <- best_model.pt 갱신
> epoch 3 | 훈련 0.4488 | 검증 0.5513
> epoch 4 | 훈련 0.4119 | 검증 0.4846
> epoch 5 | 훈련 0.3698 | 검증 0.4541  <- best_model.pt 갱신
> epoch 6 | 훈련 0.3700 | 검증 0.4792
> epoch 7 | 훈련 0.3229 | 검증 0.4944
> epoch 8 | 훈련 0.3018 | 검증 0.4769
> ```
>
> **훈련 손실은 0.90 → 0.30 으로 쭉 내려갑니다.** 그런데 `best_model.pt` 갱신은
> **epoch 5 에서 멈췄습니다.**
>
> 즉 **마지막 epoch 8 의 모델(검증 0.4769)보다 epoch 5 의 모델(검증 0.4541)이 더 좋습니다.**
> 훈련 손실만 보고 마지막 모델을 썼다면 **더 나쁜 모델을 고른 것**입니다 —
> 2교시 과적합 이야기가 여기서 그대로 이어집니다. 이것이 **조기 종료**입니다.

---

## 실습 11 — TensorBoard 로 실험 5개 비교 (12분)

> **핵심 질문**: 1교시에 옵티마이저 3종, 2교시에 정규화 3종을 돌렸습니다.
> **그 여섯 개의 곡선을 지금 한 화면에서 볼 수 있나요?** 노트북 출력은 이미 스크롤 저 위에 있습니다.

```
   matplotlib : 한 셀에서 그린 것만 보인다. 어제 실험은 사라졌다
   TensorBoard: 로그를 파일로 남긴다 -> 언제든 다시, 여러 실험을 겹쳐서
```

In [ ]:
# 셀 9 — 5개 조건을 runs/ 에 기록
from torch.utils.tensorboard import SummaryWriter

조건 = [
    ("sgd_lr0.1",      dict(opt="sgd",  lr=0.1)),
    ("sgd_lr0.01",     dict(opt="sgd",  lr=0.01)),
    ("momentum_lr0.1", dict(opt="mom",  lr=0.1)),
    ("adam_lr1e-3",    dict(opt="adam", lr=1e-3)),
    ("adam_lr0.1",     dict(opt="adam", lr=0.1)),      # ★ 일부러 발산시킨다
]

for name, cfg in 조건:
    set_seed(42)
    model = MLP().to(device)
    opt = {"sgd":  torch.optim.SGD(model.parameters(), lr=cfg["lr"]),
           "mom":  torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9),
           "adam": torch.optim.Adam(model.parameters(), lr=cfg["lr"])}[cfg["opt"]]

# ───── 실습 11 · 빈칸 ① (1줄) ─────
# 힌트 : 실험마다 폴더를 따로 주는 기록기를 만든다. runs/<실험이름>
# 확인 : runs/ 아래에 폴더 5개가 생긴다


    for epoch in range(5):
        model.train(); run = n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item()*xb.size(0); n += xb.size(0)

# ───── 실습 11 · 빈칸 ② (2줄) ─────
# 힌트 : epoch 마다 ① 훈련 손실 ② 테스트 정확도를 기록한다. 태그는 "loss/train", "accuracy/test" (2줄)
# 확인 : adam_lr0.1 만 손실이 위로 치솟는다



    writer.close()                                        # ★ 안 하면 로그가 안 써진다
    print(f"{name:16s} 기록 완료")

### 보는 방법

```powershell
# 터미널에서 (venv 활성화 상태)
tensorboard --logdir runs
# -> http://localhost:6006 을 브라우저에서 연다
```

> ⚠️ **방화벽이 6006 포트를 막으면** 위 방법이 안 됩니다.
> 그럴 때는 **아래 셀**처럼 노트북 안에서 엽니다.

In [ ]:
# 셀 10 — 방화벽이 막혔다면 노트북 안에서 연다
#   ★ Run All 로 일괄 실행할 때는 이 셀을 건너뛰세요 (서버가 뜹니다)
%load_ext tensorboard
%tensorboard --logdir runs

> **관찰 포인트 ★**: 왼쪽 목록에서 **실험을 켜고 끄면서** 곡선을 겹쳐 봅니다.
> `adam_lr0.1` 하나만 위로 치솟아 있을 겁니다 — 1교시 실습 2에서 본 *"Adam 에 0.1 을 주면 발산"* 입니다.
> **다섯 실험을 한 화면에서 비교하는 이 경험**이 오늘 실습 11의 목적입니다.

> **막히면**
>
> | 증상 | 원인 |
> |---|---|
> | 페이지가 안 열린다 | 6006 포트 방화벽. 위 `%tensorboard` 방식으로 전환 |
> | 곡선이 안 보인다 | `writer.close()` 를 안 했거나 `--logdir` 경로가 틀렸다 |
> | 실험이 하나로 합쳐진다 | `SummaryWriter` 경로를 실험마다 다르게 줬는지 확인 |
> | `No module named tensorboard` | `pip install tensorboard` |

> ⚠️ **`runs/` 를 `.gitignore` 에 추가하세요.** 로그가 쌓이면 커집니다.

> **체크포인트**: TensorBoard 화면에 5개 실험이 보이면 통과. **과제 제출물입니다.**

---

## 3교시 핵심 정리

| 개념 | 한 줄 정의 |
|------|-----------|
| **시드 고정** | `random`·`numpy`·`torch`·`cuda` 네 곳. 모델 만들기 **직전**에 |
| **GPU 비결정성** ★ | 시드를 고정해도 완전히 같지 않을 수 있다. 대신 **조건을 기록**한다 |
| **`state_dict`** ★ | 가중치만 저장. 모델 전체 저장은 경로 의존으로 깨진다 |
| **`map_location`** | GPU 에서 저장한 것을 CPU 에서 열 때 |
| **체크포인트** ★★ | 모델 + **옵티마이저** + epoch. 옵티마이저를 빼면 재개 시 손실이 튄다 |
| **`best_model.pt`** | 검증 최저점의 가중치. 2교시 조기 종료와 짝 |
| **TensorBoard** | `SummaryWriter(f"runs/{name}")` → `add_scalar` → `close()` |

### 반드시 외우기 ★

```python
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

torch.save(model.state_dict(), "models/best_model.pt")
model.load_state_dict(torch.load(path, map_location=device))

torch.save({"epoch": e, "model": model.state_dict(),
            "optimizer": opt.state_dict()}, "models/ckpt.pt")
```

### 3교시 체크리스트

- [ ] 시드 고정 전후의 가중치 차이를 확인했다
- [ ] 시드를 고정해도 GPU 에서 완전히 같지 않을 수 있는 이유를 안다 ★
- [ ] `state_dict` 만 저장하는 이유를 말할 수 있다 ★
- [ ] 5주차 가중치를 새 모델에 불러와 정확도가 같은 것을 봤다
- [ ] **커널을 재시작한 뒤 체크포인트로 학습을 재개**했다 ★★
- [ ] 체크포인트에 **옵티마이저**를 넣는 이유를 안다
- [ ] `best_model.pt` 갱신이 멈추는 지점을 봤다
- [ ] TensorBoard 에서 실험 5개를 겹쳐 봤다
- [ ] `runs/`, `models/` 를 `.gitignore` 에 추가했다

---

## 부록. 6주차를 마치며

```
┌────────────────────────────────────────────────────────────────┐
│  오늘 손에 남은 것                                              │
├────────────────────────────────────────────────────────────────┤
│  08_optimizers.ipynb          옵티마이저·스케줄러       [1교시] │
│  09_regularization.ipynb      과적합·드롭아웃·지표      [2교시] │
│  10_experiment_tracking.ipynb 시드·저장·재개·TB         [3교시] │
│  models/best_model.pt         검증 최저점의 가중치              │
│  runs/                        실험 5개의 로그                   │
└────────────────────────────────────────────────────────────────┘
```

> **오늘의 한 문장**: 성능을 올리는 것보다 **올랐다는 걸 믿을 수 있게 만드는 것**이 어렵고 중요하다 —
> 두 곡선을 함께 보고, 조건을 하나만 바꾸고, 시드를 고정하고, 기록을 남긴다.